# **Explicación de la lectura de dorsales**

La lectura de dorsales se añade como una capa adicional para obtener información que permita corregir los errores del tracker base y mejorar el seguimiento. 

Para la lectura de dorsales, se entrena un modelo de detección de dorsales. Este modelo facilita la lectura de dorsales para el OCR, dado que entonces el OCR trabaja únicamente con la región del dorsal. También se justifica el uso de un modelo de detección de dorsales.

A continuación, se analizan varios modelos de OCR para la lectura de dorsales y se escoge el que mejor resultado obtiene.

Por último, se describe cómo se asocia el número de dorsal leído a un jugador concreto.

En concreto, el notebook tiene las siguientes partes:

- **Introducción**: Introducción de los tres OCRs comparados
- **Entrenamiento del modelo de detección de dorsales**: Explicación del modelo entrenado y justificación de la necesidad de un modelo de detección de dorsales.
- **Selección de un modelo OCR**
- **Tratamiento de detecciones de dorsal para mejora del OCR**

El código para obtener las predicciones OCR está en el notebook `obtener_predicciones_OCR.ipynb` de la misma carpeta de este notebook. El código que utiliza estas predicciones para obtener las métricas y gráficas utilizadas en este notebook para seleccionar el mejor OCR se encuentran en el notebook `obtener_metricas_graficas_OCR.ipynb`, que también se encuentra en esta misma carpeta.

## **Introducción**

El reconocimiento óptico de caracteres (OCR) es una tecnología que obtiene los caracteres visibles en una imagen. En el contexto de este proyecto, se utiliza para leer los números de los dorsales de los jugadores.

En la actualidad existen múltiples modelos de OCR, tanto comerciales como de código abierto. A continuación, se describen brevemente los tres evaluados experimentalmente como candidatos para la lectura de dorsales.

- *EasyOCR*: Es una librería disponible en Python que permite realizar OCR en imágenes. El sistema de OCR utiliza varias redes neuronales para reconocer la secuencia de caracteres en la imagen.

- *Tesseract*: Es otro sistema de OCR inicialmente desarrollado por Hewlett-Packard y posteriormente mantenido por Google. Está disponible el software para su instalación local, y posteriormente se puede utilizar en Python a través de la librería pytesseract.

- *PARSeq*: es otro sistema de OCR disponible en Python, que se puede descargar utilizando la librería torch. Su funcionamiento se describe en detalle en [este paper](https://doi.org/10.1007/978-3-031-19815-1_11).

## **Entrenamiento del modelo de detección de dorsales**

### **Conjunto de datos utilizado**

Para la lectura de dorsales, se ha entrenado un modelo capaz de detectar los números de dorsal en las imágenes. Para ello, se ha utilizado [este dataset](https://universe.roboflow.com/roboflow-jvuqo/basketball-player-detection-3-ycjdo).

Este dataset tiene las siguientes características:
- **Número de imágenes**: El dataset está compuesto por 654 imágenes.
- **Clases**: El dataset contiene las clases *ball* (la pelota), *ball-in-basket* (la pelota dentro de la canasta), *number* (el dorsal), *player* (el jugador), *player-in-possession* (jugador con la pelota), *player-jump-shot* (jugador lanzando mientras salta), *player-layup-dunk* (jugador encestando desde muy cerca del aro), *player-shot-block* (jugador bloqueando un lanzamiento a canasta), *referee* (árbitro), y *rim* (aro)
-  **Aumentación de datos**: El dataset no se ha aumentado.
-  **Tipo de imágenes**: Imágenes desde los dos lados de la cancha, con dos tipos de cancha distintas.
-  **Conjuntos de entrenamiento, validación y test**: El dataset cuenta con un 71\% de imágenes para entrenamiento (464), un 15\% para validación (96), y un 14\% para test (94).

El modelo utilizado es YOLO, dado que es un modelo de detección de objetos en tiempo real, lo que es adecuado para el caso de uso del trabajo. Se ha utilizado la versión 8 mediana de YOLO, dado que presenta un buen equilibrio entre precisión y velocidad.

Dado que un número excesivo de clases podría dificultar el entrenamiento, se agrupan las clases referidas a jugadores (*player*, *player-in-possession*, *player-jump-shot*, *player-layup-dunk*, y *player-shot-block*) en una única clase *player*. Además, se mantiene la clase *number* para la detección de dorsales, y la clase *ball* porque se considera de utilidad para añadir métricas adicionales de los jugadores.

### **Entrenamiento del modelo**

El modelo se entrena durante *100 épocas*, con una paciencia de *30 épocas* y un tamaño de batch de *8*.

Respecto al recall, los jugadores obtienen un recall de 0.98, los dorsales de 0.91 y la pelota de 0.83. Dado que se detectan los dorsales en la mayoría de frames, un recall de 0.91 se considera suficiente. Respecto a la pelota, se observa que no se detecta en algunos frames. Sin embargo, es posible obtener información útil de la pelota en un número suficiente de frames, y no es el foco principal del trabajo, por lo que un recall de 0.83 se considera aceptable.

Tras este entrenamiento, se probó a utilizar este modelo como detector de jugadores para los trackers, pero se observó que el modelo de detección de jugadores entrenado en el TFG obtenía mejores resultados. Esto podría deberse a que el modelo anterior se entrenó con unas 1500 imágenes y con mayor variabilidad en ángulos de la cámara, iluminación, y campos, lo que le permite generalizar mejor. Sin embargo, la disponibilidad de conjuntos de datos para detección de dorsales es mucho más limitada, por lo que se utiliza este dataset para la detección de dorsales.

### **Justificación de un detector de dorsales**

Tras realizar pruebas con diversos frames, se observa de forma consistente que, mientras Parseq lee correctamente los *bounding boxes* de los dorsales, ninguno de los 3 OCRs es capaz de leer consistentemente los *bounding boxes* de los jugadores.

Se ha probado a recortar la caja de jugadores de diversas formas, con el fin de que el OCR lea mejor la región del dorsal, pero esto no mejora los resultados, y si se realiza un recorte mayor, se dejan parcialmente fuera algunos dorsales, empeorando las lecturas. 

Estas observaciones justifican la necesidad de obtener un modelo de detección de dorsales.

## **Selección de un modelo de OCR**

### **Conjunto de datos utilizado en la comparativa**

Para la comparación, se utiliza [este dataset](https://universe.roboflow.com/roboflow-jvuqo/basketball-jersey-numbers-ocr), que contiene 3600 imágenes de dorsales de jugadores de baloncesto, junto con sus respectivos números de dorsal. Este dataset contiene imágenes de los números 0 a 40, con variedad de colores de números y uniformes. El dataset contiene 3667 imágenes, de las cuales quedan 3615 tras eliminar aquellas con número no anotado.

En la siguiente figura se muestran algunos ejemplos de imágenes del dataset utilizado.

<figure style="text-align: center; margin: 1.5em 0;">
  <div style="display: flex; justify-content: space-between; align-items: center; gap: 6px;">
    <img src="../img/dataset/ocr/1.jpg" alt="1" style="width: 11%; height: auto;" />
    <img src="../img/dataset/ocr/2.jpg" alt="2" style="width: 11%; height: auto;" />
    <img src="../img/dataset/ocr/3.jpg" alt="3" style="width: 11%; height: auto;" />
    <img src="../img/dataset/ocr/4.jpg" alt="4" style="width: 11%; height: auto;" />
    <img src="../img/dataset/ocr/5.jpg" alt="5" style="width: 11%; height: auto;" />
    <img src="../img/dataset/ocr/6.jpg" alt="6" style="width: 11%; height: auto;" />
    <img src="../img/dataset/ocr/7.jpg" alt="7" style="width: 11%; height: auto;" />
    <img src="../img/dataset/ocr/8.jpg" alt="8" style="width: 11%; height: auto;" />
  </div>
  <figcaption style="margin-top: 10px; font-size: 0.9em; color: #555;">
    Ejemplos de imágenes del dataset de OCR de dorsales
  </figcaption>
</figure>

### **Tratamiento de salida de OCR**

Dado que tesseract y easyocr devuelven varias cadenas de caracteres, se considera que el número de OCR es la cadena que sólo contenga números. Si existen varias cadenas que sólo contengan números, se escoge la que mayor confianza tenga. Si no existe ninguna cadena que sólo contenga números, se considera que el OCR no ha leído ningún número (cadena vacía). Esto se realiza para un análisis más justo, dado que si se juntan todas las cadenas de caracteres, los resultados de los dos OCRs empeoran.

Para Parseq, se considera la cadena de caracteres que devuelve el modelo, y como confianza se considera la confianza mínima de los caracteres que componen la cadena, dado que lo relevante es que todos los caracteres del número de dorsal se lean correctamente.

### **Comparativa de OCRs**

Para las métricas, se obtienen métricas considerando números a partir de una confianza dada y que su predicción no sea vacía. De este modo, se calcula el porcentaje de imágenes consideradas para cada valor de confianza, y el porcentaje de aciertos sobre las imágenes consideradas.

En la siguiente figura se muestra la comparativa de los tres OCRs en una gráfica con los aciertos en azul y las imágenes consideradas en naranja.

<figure style="text-align: center; margin: 1.5em 0;">
  <div style="display: flex; justify-content: space-between; align-items: flex-start; gap: 12px;">
    <div style="width: 32%; text-align: center;">
      <img src="../img/comparativa_OCR/tesseract.png" alt="Tesseract" style="width: 100%; height: auto;" />
      <div style="margin-top: 6px; font-size: 0.85em; color: #444;">(a) Tesseract</div>
    </div>
    <div style="width: 32%; text-align: center;">
      <img src="../img/comparativa_OCR/easyocr.png" alt="EasyOCR" style="width: 100%; height: auto;" />
      <div style="margin-top: 6px; font-size: 0.85em; color: #444;">(b) EasyOCR</div>
    </div>
    <div style="width: 32%; text-align: center;">
      <img src="../img/comparativa_OCR/parseq.png" alt="Parseq" style="width: 100%; height: auto;" />
      <div style="margin-top: 6px; font-size: 0.85em; color: #444;">(c) Parseq</div>
    </div>
  </div>
  <figcaption style="margin-top: 10px; font-size: 0.9em; color: #555;">
    Comparativa de Tesseract, EasyOCR y Parseq para lectura de dorsales
  </figcaption>
</figure>

Respecto a **Tesseract**, se observa que el *porcentaje de imágenes consideradas es menor a 0.1*, por lo que no se considera un modelo adecuado para la lectura de dorsales. 

Respecto a **EasyOCR**, se obtienen resultados para un mayor porcentaje de imágenes. Con un *umbral de confianza 0 se descarta el 40\% de las imágenes*, de modo que el modelo no presenta detección para estas imágenes.

A medida que el umbral de confianza aumenta, el porcentaje de imágenes consideradas disminuye, hasta menos del 40\% para un umbral de confianza de 0.95. A su vez, el porcentaje de aciertos aumenta, desde menos del 80\% para un umbral de confianza de 0, hasta poco más del 90\% con un umbral de 0.95. 

Cuando se consideran sólo imágenes con confianza de 1, el acierto es total pero se considera un porcentaje cercano al 0\%

**Parseq** es el modelo que mejores resultados obtiene. Para un *umbral de confianza de 0, se obtiene predicción para el 86\% de las imágenes*, con un porcentaje de aciertos del 90\%. A medida que el umbral de confianza aumenta, el porcentaje de imágenes consideradas disminuye y el porcentaje de aciertos aumenta, hasta obtener un acierto del 98\% y un porcentaje de imágenes del 57\% para un umbral de confianza de 0.95.

### **No se busca 100\% de respuesta en el conjunto de datos**

Es relevante destacar que, aunque la mayoría de números de dorsal son reconocibles, existen algunas instancias del dataset en las que el número es facilmente confundible, como se muestra en la siguiente figura.

<figure style="text-align: center; margin: 1.5em 0;">
  <div style="display: flex; justify-content: space-between; align-items: center; gap: 6px;">
    <img src="../img/comparativa_OCR/oklahoma-city-thunder-denver-nuggets-game-1-q3-08_19-08_10-0210-0003_png.rf.01e8748dec9552dec77731c5683fe58c.jpg" alt="ejemplo 1" style="width: 11%; height: auto;" />
    <img src="../img/comparativa_OCR/oklahoma-city-thunder-denver-nuggets-game-1-q4-09_09-09_04-0150-0004_png.rf.0a12b3c04e76c0cdbf746603011fc2fa.jpg" alt="ejemplo 2" style="width: 11%; height: auto;" />
    <img src="../img/comparativa_OCR/oklahoma-city-thunder-memphis-grizzlies-game-4-q2-06_34-06_27-0000-0000_png.rf.d99ee5e555d2308c01bf6d9267191e07.jpg" alt="ejemplo 3" style="width: 11%; height: auto;" />
    <img src="../img/comparativa_OCR/boston-celtics-new-york-knicks-game-4-q1-01_22-01_16-0030-0001_png.rf.e56b2e20e9fb79d2e2f4734535496165.jpg" alt="ejemplo 4" style="width: 11%; height: auto;" />
    <img src="../img/comparativa_OCR/denver-nuggets-los-angeles-clippers-game-1-q1-09_52-09_44-0000-0000_png.rf.384b062f2363b6ae99c70f89ad422542.jpg" alt="ejemplo 5" style="width: 11%; height: auto;" />
    <img src="../img/comparativa_OCR/denver-nuggets-los-angeles-clippers-game-1-q1-09_52-09_44-0150-0004_png.rf.fce3100258bb8839347f8660467c1991.jpg" alt="ejemplo 6" style="width: 11%; height: auto;" />
    <img src="../img/comparativa_OCR/denver-nuggets-los-angeles-clippers-game-1-q1-09_52-09_44-0090-0001_png.rf.012659495d648eedbb5e2078046e0efd.jpg" alt="ejemplo 7" style="width: 11%; height: auto;" />
    <img src="../img/comparativa_OCR/new-york-knicks-detroit-pistons-game-4-q1-00_31-00_09-0000-0001_png.rf.bb7a4636af5f4ec8513f45a31473f31d.jpg" alt="ejemplo 8" style="width: 11%; height: auto;" />
  </div>
  <figcaption style="margin-top: 10px; font-size: 0.9em; color: #555;">
    Ejemplos de imágenes del dataset de OCR de dorsales con número de dorsal difícil de leer
  </figcaption>
</figure>

Estas imágenes no se eliminarán porque es relevante que el modelo sea capaz de gestionar estos casos, pero dado que existen este tipo de imágenes, no se busca un modelo que de respuesta para un 100\% de las imágenes de forma correcta, sino un modelo que sea capaz de dar respuesta a un porcentaje elevado de los dorsales y que su porcentaje de acierto sea elevado.

### **Rendimiento consistente del OCR**

Se observa el número de aciertos para cada número de dorsal sobre todas las imágenes del conjunto de datos para el OCR Parseq, dado que es el modelo con mejores resultados, con el fin de determinar si su rendimiento es consistente para todos los números.

Se obtiene un porcentaje de imágenes no consideradas, porque el OCR no lee el número, de entre 0 y 30\%, a la vez que el porcentaje de aciertos sobre las imágenes consideradas varía entre el 71\% y el 100\%.

Es importante tener en cuenta que el número de dorsal con porcentaje de acierto del 71\% sólo dispone de 8 imágenes, dado que no todos los números de dorsal tiene el mismo número de imágenes.

Se observa que **Parseq** obtiene buenos resultados y no se comporta de forma muy diferente para los distintos números de dorsal, por lo que **se selecciona para la lectura de dorsales**.

## **Tratamiento de detecciones de dorsal para mejora del OCR**

Se propone como *hipótesis* que *ampliar la región del dorsal detectada* por el modelo de detección de dorsales puede *mejorar la lectura* del número. 

Mediante pruebas manuales por no disponer de un conjunto de datos específico para esta prueba, se prueba a *expandir* el *bounding box* del dorsal un *50\%* tanto en altura como en anchura con el fin de garantizar que el número quede completamente incluido en el recorte. 

Se observa que este aumento *mejora la lectura* de algunos dorsales detectados con una localización no tan precisa. Por lo tanto, se opta por aplicar esta medida en el sistema.